In [1]:
#importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import random

In [ ]:
# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# Paths
base_dir = 'drive/MyDrive/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Data Generator
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# Image size and batch
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load Train & Validation
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

# Load Test Data
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 25919 images belonging to 3 classes.
Found 6479 images belonging to 3 classes.
Found 10500 images belonging to 3 classes.


In [ ]:

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Input shape
IMG_SHAPE = (224, 224, 3)


base_model = VGG16(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)
base_model.trainable = False  # Freeze base layers for initial training

# Add custom classification head
inputs = Input(shape=IMG_SHAPE)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(3, activation='softmax')(x)

vgg16_model = Model(inputs, outputs)

# Compile the model

vgg16_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Summary of the model

vgg16_model.summary()

# Train the model (initial training phase with frozen base)

history_vgg16 = vgg16_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    verbose=1
)

In [ ]:
# Evaluate on the test set

test_loss, test_acc = vgg16_model.evaluate(test_data)
print(f"\n Test Accuracy: {test_acc * 100:.2f}%")
print(f" Test Loss: {test_loss:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tensorflow.keras.utils import to_categorical

# Predict probabilities

y_probs = vgg16_model.predict(test_data)
y_pred = np.argmax(y_probs, axis=1)
y_true = test_data.classes

# For ROC-AUC
y_true_categorical = to_categorical(y_true, num_classes=3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Classification report
print(" Classification Report:")
print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_data.class_indices.keys())

plt.figure(figsize=(6, 5))
disp.plot(cmap="Blues", values_format='d')
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()

In [ ]:
from sklearn.metrics import auc

fpr = {}
tpr = {}
roc_auc = {}
class_names = list(test_data.class_indices.keys())

for i in range(3):
    fpr[i], tpr[i], _ = roc_curve(y_true_categorical[:, i], y_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC AUC
plt.figure(figsize=(8, 6))
colors = ['blue', 'green', 'red']

for i in range(3):
    plt.plot(fpr[i], tpr[i], color=colors[i], label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('📊 ROC-AUC Curves by Class')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot Accuracy and Loss (Training + Validation)
# def plot_training_metrics(history, title_prefix="MobileNetV2"):
def plot_training_metrics(history, title_prefix="VGG16"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    plt.figure(figsize=(14, 5))

    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy', marker='o')
    plt.plot(val_acc, label='Validation Accuracy', marker='o')
    plt.title(f'{title_prefix} - Training vs Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()

    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss', marker='o')
    plt.plot(val_loss, label='Validation Loss', marker='o')
    plt.title(f'{title_prefix} - Training vs Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Use this with your latest training history

plot_training_metrics(history_vgg16)